In [ ]:
import os, re, json, math, time, warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import rankdata
from sklearn.metrics import roc_auc_score, average_precision_score, roc_curve

warnings.filterwarnings("ignore", category=FutureWarning)

# ---------------- CONFIG ----------------
INPUT_ROOT = Path(os.environ.get("MERGE_INPUT", "/kaggle/input"))
OUT = Path(os.environ.get("MERGE_OUT", "/kaggle/working"))
FIG = OUT / "figs"
FIG.mkdir(parents=True, exist_ok=True)

EXCLUDE_BENCH = {"aigibench"}
BENCH_ORDER = ["itw_sm", "wildrf", "bfree_viral_images"]
BENCH_LABEL = {"itw_sm": "ITW-SM", "wildrf": "WildRF", "bfree_viral_images": "B-Free viral"}
EXPECTED_N = {"itw_sm": 10000, "wildrf": 2503, "bfree_viral_images": 785}   # chỉ để cảnh báo
MIN_COVER = 0.99          # model phải chấm >= 99% số ảnh của mỗi benchmark mới vào bảng chính

OURS = ["s7_seed42", "s7_seed54", "s7_seed68"]
OFFICIAL = "bfree_official"
# baseline = mọi model_id còn lại tìm thấy trong scores_*.parquet (stage-2)
MODEL_LABEL = {
    "s7_seed42": "Ours (seed 42)", "s7_seed54": "Ours (seed 54)", "s7_seed68": "Ours (seed 68)",
    "bfree_official": "B-Free (official)",
    "dda": "DDA", "commfor384": "CommFor (ViT-S/16-384)", "gapl": "GAPL",
    "globalforge_dinov3": "GlobalForge (DINOv3)", "globalforge_dinov2": "GlobalForge (DINOv2)",
    "cospy": "Co-Spy", "effort": "Effort", "c2p_clip": "C2P-CLIP", "safe": "SAFE",
    "aigi_holmes": "AIGI-Holmes",
}
# ghi chú protocol hiển thị kèm trong slide_notes.md (sửa tay nếu đã fix)
PROTOCOL_CAVEATS = {
    "safe": "stage-2 chạy fp16 autocast trong khi loader SAFE khuyến nghị fp32 (DWT lệch dtype) -> chạy lại fp32 trước khi kết luận",
}

N_BOOT = 300           # bootstrap CI AUC của từng model/subset
N_BOOT_PAIRED = 1000   # bootstrap cặp: AUC(ours, TB các seed) - AUC(ref)
SEED = 42
DEC = 3                # số chữ số thập phân trong bảng slide

C_OURS, C_OFF, C_BASE = "#10346b", "#d9731a", "#8a97a8"
C_SEED = ["#1b4f9c", "#4f86d6", "#93b5e8"]
BASE_PALETTE = ["#2e7d4f", "#8e44ad", "#c0392b", "#16a085", "#b7950b", "#5d6d7e", "#a04000", "#117a65", "#6c3483"]
plt.rcParams.update({"figure.facecolor": "white", "axes.facecolor": "white", "axes.spines.top": False,
                     "axes.spines.right": False, "axes.grid": True, "grid.alpha": 0.25, "font.size": 10,
                     "axes.titleweight": "bold", "figure.dpi": 110})


def label_of(m):
    return MODEL_LABEL.get(m, m)


## 1. Đọc dữ liệu và kiểm tra

In [ ]:
# 1) Đọc điểm (scores_*.parquet), bỏ AIGIBench ngay khi đọc, ghi nhận nguồn (stage) của từng model
pat = re.compile(r"^scores_(?P<mid>.+)_(?:nb|shard)(?P<k>\d+)\.parquet$")
stage_pat = re.compile(r"stage[-_ ]?(\d+)", re.I)


def stage_of_path(p):
    for part in reversed(Path(p).parts):
        m = stage_pat.search(part)
        if m:
            return f"stage-{m[1]}"
    return "unknown"


found = []
for root, _, files in os.walk(INPUT_ROOT):
    for f in files:
        m = pat.match(f)
        if m:
            found.append((Path(root) / f, m["mid"]))
if not found:
    raise FileNotFoundError(f"Không thấy scores_*.parquet nào dưới {INPUT_ROOT} - attach output của notebook eval gốc")

frames, dropped, PROV = [], 0, []
for p, mid in sorted(found):
    d = pd.read_parquet(p)
    if "logit" not in d.columns:
        s = d["score"].clip(1e-9, 1 - 1e-9)
        d["logit"] = np.log(s / (1 - s))
    if "condition" in d.columns:
        d = d[d["condition"] == "clean"]
    keep = ~d["benchmark"].isin(EXCLUDE_BENCH)
    dropped += int((~keep).sum())
    d = d[keep][["id", "label", "logit", "benchmark", "generator"]].copy()
    if len(d):
        d["model_id"] = mid
        d["stage"] = stage_of_path(p)
        frames.append(d)
        PROV.append({"model_id": mid, "stage": stage_of_path(p), "folder": p.parent.name, "file": p.name, "rows": len(d)})
        print(f"{p.parent.name[-34:]:<35} {p.name:<44} {len(d):>7} dòng giữ lại")
    else:
        print(f"{p.parent.name[-34:]:<35} {p.name:<44} (toàn bộ là benchmark bị loại -> bỏ qua)")
if not frames:
    raise ValueError("Không còn dữ liệu sau khi loại benchmark - attach output của notebook eval gốc")

S = pd.concat(frames, ignore_index=True)
S["label"] = S["label"].astype(int)
S["generator"] = S["generator"].astype(str)
n0 = len(S)
S = S[np.isfinite(S["logit"])]
dups = S.duplicated(["model_id", "id"]).sum()
if dups:
    print(f"CẢNH BÁO: {dups} dòng trùng (model, id) - giữ bản cuối (có thể attach trùng 2 phiên bản output)")
    S = S.drop_duplicates(["model_id", "id"], keep="last")
BENCH = [b for b in BENCH_ORDER if b in S.benchmark.unique()]
STAGE_OF = S.groupby("model_id")["stage"].agg(lambda x: "/".join(sorted(set(x)))).to_dict()
print(f"\nĐã loại {dropped} dòng thuộc {sorted(EXCLUDE_BENCH)} | giữ {len(S)} dòng ({n0 - len(S)} loại: NaN/trùng) | benchmark: {BENCH}")
print("Stage của từng model:", {m: STAGE_OF[m] for m in sorted(STAGE_OF)})


In [ ]:
# 2) Độ đầy đủ: model nào chấm đủ ảnh -> bảng chính; thiếu -> PARTIAL (tách riêng)
cnt = S.groupby(["model_id", "benchmark"]).size().unstack("benchmark").reindex(columns=BENCH)
ref_n = {b: int(cnt[b].max()) for b in BENCH}
for b in BENCH:
    if ref_n[b] != EXPECTED_N[b]:
        print(f"CẢNH BÁO: {b} tối đa {ref_n[b]} ảnh, kỳ vọng {EXPECTED_N[b]}")
full = [m for m in cnt.index if all(pd.notna(cnt.loc[m, b]) and cnt.loc[m, b] >= MIN_COVER * ref_n[b] for b in BENCH)]
PARTIAL = [m for m in cnt.index if m not in full]
others = [m for m in full if m not in OURS and m != OFFICIAL]
MODELS = [m for m in OURS if m in full] + ([OFFICIAL] if OFFICIAL in full else []) + sorted(others)
OUR_COLS = [m for m in OURS if m in MODELS]
miss = [m for m in OURS + [OFFICIAL] if m not in MODELS]
print(cnt.fillna(0).astype(int).to_string())
print(f"\nModel đủ ảnh (bảng chính): {MODELS}")
if PARTIAL:
    print(f"Model chấm chưa đủ ảnh (tách riêng, không so sánh trực tiếp): {PARTIAL}")
if miss:
    print("CẢNH BÁO: thiếu model của bạn/official:", miss)
if not OUR_COLS:
    print("CẢNH BÁO: không có seed nào của Ours -> bảng chỉ liệt kê baseline, bỏ qua so sánh cặp")

# 3) Căn chỉnh id giữa các stage (cần để bootstrap CẶP trên cùng tập ảnh)
ANCHOR = (OUR_COLS + ([OFFICIAL] if OFFICIAL in MODELS else []) + MODELS)[0]
S["key"] = S["id"]


def _overlap(col):
    worst = 1.0
    for b in BENCH:
        base = set(S.loc[(S.model_id == ANCHOR) & (S.benchmark == b), col])
        for m in MODELS:
            if m == ANCHOR:
                continue
            ks = set(S.loc[(S.model_id == m) & (S.benchmark == b), col])
            worst = min(worst, len(base & ks) / max(len(base), 1))
    return worst


worst = _overlap("key")
PAIRED_OK = worst >= 0.99
if not PAIRED_OK:
    alt = (S.benchmark + "|" + S.generator.str.lower() + "|" + S.label.astype(str) + "|" +
           S.id.str.replace("\\", "/", regex=False).str.split("/").str[-1])
    S["_alt"] = alt
    no_dup = not S.duplicated(["model_id", "_alt"]).any()
    worst_alt = _overlap("_alt") if no_dup else 0.0
    if worst_alt >= 0.99:
        S["key"] = S["_alt"]
        PAIRED_OK, worst = True, worst_alt
        print(f"id khác nhau giữa các stage -> dùng khóa phụ (benchmark|generator|label|tên file), overlap tối thiểu {worst:.4f}")
    else:
        print(f"CẢNH BÁO: id không khớp giữa các stage (overlap tối thiểu {worst:.3f}, khóa phụ {worst_alt:.3f}) "
              "-> BỎ bootstrap cặp; bảng AUC vẫn tính trên ảnh riêng của từng model")
else:
    print(f"id khớp giữa các model (overlap tối thiểu {worst:.4f})")

# bảng rộng: mỗi hàng = 1 ảnh, mỗi cột = logit của 1 model
W = pd.DataFrame()
if PAIRED_OK:
    Sm = S[S.model_id.isin(MODELS)]
    bad_lab = Sm.groupby("key")["label"].nunique()
    bad_lab = bad_lab[bad_lab > 1].index
    if len(bad_lab):
        print(f"CẢNH BÁO: {len(bad_lab)} ảnh có nhãn khác nhau giữa các model -> loại")
        Sm = Sm[~Sm.key.isin(bad_lab)]
    W = Sm.pivot_table(index="key", columns="model_id", values="logit", aggfunc="last")[MODELS]
    meta = Sm[Sm.model_id == ANCHOR].drop_duplicates("key").set_index("key")[["benchmark", "generator", "label"]]
    W = W.join(meta, how="inner")
    n_inc = int(W[MODELS].isna().any(axis=1).sum())
    if n_inc:
        print(f"CẢNH BÁO: {n_inc} ảnh thiếu điểm ở ít nhất 1 model -> loại khỏi phần so sánh cặp")
    W = W.dropna(subset=MODELS).reset_index().set_index(["benchmark", "generator", "key", "label"])[MODELS]
    print("Số ảnh dùng cho so sánh cặp:", W.groupby(level="benchmark").size().to_dict())


## 2. Metric

In [ ]:
# 3) Metric: AUC (+CI bootstrap), AP, TPR/TNR tại ngưỡng logit=0 (score 0.5), bacc, EER
def auc_fast(y, z):
    r = rankdata(z)
    n1 = int(y.sum()); n0 = len(y) - n1
    return (r[y == 1].sum() - n1 * (n1 + 1) / 2) / (n1 * n0)


def auc_cols(y, Z):
    # AUC cho từng cột của Z (rank theo trục 0), cùng nhãn y
    r = rankdata(Z, axis=0)
    n1 = int(y.sum()); n0 = len(y) - n1
    return (r[y == 1].sum(axis=0) - n1 * (n1 + 1) / 2) / (n1 * n0)


def auc_ci(y, z, n_boot, rng):
    pos, neg = z[y == 1], z[y == 0]
    out = np.empty(n_boot)
    for b in range(n_boot):
        p, n = rng.choice(pos, len(pos)), rng.choice(neg, len(neg))
        out[b] = auc_fast(np.r_[np.ones(len(p), int), np.zeros(len(n), int)], np.concatenate([p, n]))
    return np.percentile(out, [2.5, 97.5])


def metrics(y, z, ci=True):
    y, z = np.asarray(y, int), np.asarray(z, float)
    if y.min() == y.max():
        return None
    pred = z > 0
    tpr, tnr = pred[y == 1].mean(), (~pred[y == 0]).mean()
    fpr, tp, _ = roc_curve(y, z)
    fnr = 1 - tp
    i = np.nanargmin(np.abs(fnr - fpr))
    m = {"n": len(y), "n_real": int((y == 0).sum()), "n_fake": int((y == 1).sum()),
         "auc": roc_auc_score(y, z), "ap": average_precision_score(y, z), "acc": (pred == y).mean(),
         "tpr_fake": tpr, "tnr_real": tnr, "bacc": (tpr + tnr) / 2, "eer": (fpr[i] + fnr[i]) / 2}
    if ci:
        m["auc_lo"], m["auc_hi"] = auc_ci(y, z, N_BOOT, np.random.default_rng(SEED))
    return m


t0 = time.time()
rows, pooled = [], []
for (b, g, mid), d in S.groupby(["benchmark", "generator", "model_id"]):
    m = metrics(d.label.values, d.logit.values)
    if m is None:
        print(f"bỏ qua {b}/{g}/{mid}: chỉ có 1 lớp")
        continue
    rows.append({"benchmark": b, "generator": g, "model_id": mid, **m})
PG = pd.DataFrame(rows)
for (b, mid), d in S.groupby(["benchmark", "model_id"]):
    m = metrics(d.label.values, d.logit.values)
    if m:
        pooled.append({"benchmark": b, "model_id": mid, **m})
PO = pd.DataFrame(pooled)

NUM = ["auc", "ap", "acc", "tpr_fake", "tnr_real", "bacc", "eer"]
MA = PG.groupby(["benchmark", "model_id"])[NUM].mean().reset_index()      # macro: trung bình theo subset
print(f"metric xong trong {time.time() - t0:.0f}s")

# model partial: metric trên các ảnh đã chấm (chỉ để tham khảo)
if PARTIAL:
    PP = PO[PO.model_id.isin(PARTIAL)][["model_id", "benchmark", "n", "auc", "auc_lo", "auc_hi", "bacc", "tpr_fake", "tnr_real"]]
    PP.to_csv(OUT / "partial_models.csv", index=False)
    print("\nModel chưa chấm đủ ảnh (tham khảo, KHÔNG so sánh trực tiếp với bảng chính):")
    print(PP.round(4).to_string(index=False))

# thứ tự hàng: Ours (TB seed) -> official -> baseline xếp theo AUC pooled trung bình giảm dần
_avg = PO[PO.model_id.isin(MODELS)].pivot(index="model_id", columns="benchmark", values="auc")[BENCH].mean(axis=1)
BASE_IDS = sorted([m for m in MODELS if m not in OURS and m != OFFICIAL], key=lambda m: -_avg.get(m, -1))
ROW_IDS = (["ours"] if OUR_COLS else []) + ([OFFICIAL] if OFFICIAL in MODELS else []) + BASE_IDS
STAGE_OF["ours"] = STAGE_OF.get(OUR_COLS[0], "-") if OUR_COLS else "-"
OURS_LABEL = f"Ours (mean ± std, {len(OUR_COLS)} seeds)"
MCOL = {"ours": C_OURS, OFFICIAL: C_OFF}
MCOL.update({m: c for m, c in zip(BASE_IDS, BASE_PALETTE * 3)})
MCOL.update({m: c for m, c in zip(OUR_COLS, C_SEED)})


def row_label(r):
    return OURS_LABEL if r == "ours" else label_of(r)


print(PO[PO.model_id.isin(MODELS)].pivot(index="model_id", columns="benchmark", values="auc")[BENCH]
      .reindex(MODELS).round(4).to_string())


## 3. Bootstrap cặp: Ours (TB các seed) so với từng model tham chiếu

In [ ]:
# 4) Bootstrap CẶP cho chênh lệch AUC = mean(AUC ours các seed) - AUC(ref), với ref = official + mọi baseline.
#    Resample CÙNG tập ảnh cho mọi model (phân tầng real/fake, theo subset). CI chỉ phản ánh biến động do chọn ảnh;
#    biến động giữa các seed hiển thị riêng (± std ở bảng).
REF_COLS = [m for m in MODELS if m not in OURS]
COLS = OUR_COLS + REF_COLS
nO = len(OUR_COLS)


def bh(p):
    p = np.asarray(p, float); m = len(p)
    o = np.argsort(p)
    q = np.minimum.accumulate((p[o] * m / (np.arange(m) + 1))[::-1])[::-1]
    out = np.empty(m); out[o] = np.clip(q, 0, 1)
    return out


def boot_p(d):
    d = np.asarray(d)
    return min(1.0, 2 * min((d <= 0).sum() + 1, (d >= 0).sum() + 1) / (len(d) + 1))


def _verdict(lo, hi):
    return "ours better" if lo > 0 else ("ours worse" if hi < 0 else "n.s.")


PGp = PG.pivot_table(index=["benchmark", "generator"], columns="model_id", values="auc")
POp = PO.pivot_table(index="benchmark", columns="model_id", values="auc")
PB = pd.DataFrame()
if PAIRED_OK and OUR_COLS and REF_COLS:
    t0 = time.time()
    R = len(REF_COLS)
    recs = []
    for b in BENCH:
        groups = []
        for g, Wg in W.loc[b].groupby(level="generator"):
            y = Wg.index.get_level_values("label").values.astype(int)
            groups.append((g, y, Wg[COLS].values))
        G = len(groups)
        rng = np.random.default_rng(SEED)
        D = np.empty((N_BOOT_PAIRED, G, R)); P = np.empty((N_BOOT_PAIRED, R))
        pidx = [(np.where(y == 1)[0], np.where(y == 0)[0]) for _, y, _ in groups]
        for it in range(N_BOOT_PAIRED):
            ys, Zs = [], []
            for k, (g, y, Z) in enumerate(groups):
                pos, neg = pidx[k]
                idx = np.r_[rng.choice(pos, len(pos)), rng.choice(neg, len(neg))]
                yy, ZZ = y[idx], Z[idx]
                a = auc_cols(yy, ZZ)
                D[it, k, :] = a[:nO].mean() - a[nO:]
                if G > 1:
                    ys.append(yy); Zs.append(ZZ)
            if G > 1:
                a = auc_cols(np.concatenate(ys), np.vstack(Zs))
                P[it] = a[:nO].mean() - a[nO:]
            else:
                P[it] = D[it, 0]
        for r, ref in enumerate(REF_COLS):
            for k, (g, _, _) in enumerate(groups):
                lo, hi = np.percentile(D[:, k, r], [2.5, 97.5])
                obs = PGp.loc[(b, g), OUR_COLS].mean() - PGp.loc[(b, g), ref]
                recs.append({"ref": ref, "level": "subset", "benchmark": b, "generator": g, "d_auc": obs,
                             "d_lo": lo, "d_hi": hi, "p_boot": boot_p(D[:, k, r])})
            M = D[:, :, r].mean(axis=1)
            obs_m = np.mean([PGp.loc[(b, g), OUR_COLS].mean() - PGp.loc[(b, g), ref] for g, _, _ in groups])
            recs.append({"ref": ref, "level": "macro", "benchmark": b, "generator": "MACRO", "d_auc": obs_m,
                         "d_lo": np.percentile(M, 2.5), "d_hi": np.percentile(M, 97.5), "p_boot": boot_p(M)})
            obs_p = POp.loc[b, OUR_COLS].mean() - POp.loc[b, ref]
            recs.append({"ref": ref, "level": "pooled", "benchmark": b, "generator": "POOLED", "d_auc": obs_p,
                         "d_lo": np.percentile(P[:, r], 2.5), "d_hi": np.percentile(P[:, r], 97.5), "p_boot": boot_p(P[:, r])})
    PB = pd.DataFrame(recs)
    PB["q_bh"] = np.nan
    for ref in REF_COLS:                      # BH trên các test mức subset, riêng từng ref
        sel = (PB.ref == ref) & (PB.level == "subset")
        PB.loc[sel, "q_bh"] = bh(PB.loc[sel, "p_boot"].values)
    PB["verdict"] = [_verdict(lo, hi) for lo, hi in zip(PB.d_lo, PB.d_hi)]
    PB.to_csv(OUT / "paired_bootstrap_auc.csv", index=False)
    print(f"bootstrap cặp xong trong {time.time() - t0:.0f}s ({len(REF_COLS)} ref x {len(BENCH)} benchmark)")
    show = PB[PB.level == "pooled"].pivot(index="ref", columns="benchmark", values="d_auc")[BENCH].reindex(REF_COLS)
    print("\nΔAUC pooled (Ours - ref):"); print(show.round(4).to_string())
else:
    print("Bỏ qua bootstrap cặp (thiếu seed của Ours / không có ref / id không khớp giữa stage)")

PG.to_csv(OUT / "metrics_per_subset.csv", index=False)
PO.to_csv(OUT / "metrics_pooled.csv", index=False)
MA.to_csv(OUT / "metrics_macro.csv", index=False)


## 4. Bảng cho slide (CSV + Markdown + PNG + Excel)

In [ ]:
# 5) Bảng slide. Mỗi bảng: CSV (chuỗi đã format), dòng in đậm trong Markdown, PNG dán thẳng vào slide
TABLES = {}
MD_PARTS = []


def fmt_cell(v, s=np.nan, dec=DEC):
    if pd.isna(v):
        return "-"
    return f"{v:.{dec}f}" + (f" ± {s:.{dec}f}" if pd.notna(s) else "")


def with_ours(p):
    """p: index=model_id, columns=cột số. Trả (mean, std): hàng 'ours' = TB/std các seed, giữ official + baseline."""
    p = p.loc[[m for m in p.index if m in MODELS]].copy()
    p["Avg"] = p[[c for c in p.columns if c != "Avg"]].mean(axis=1)
    mean = p.loc[[m for m in p.index if m not in OUR_COLS]].copy()
    std = pd.DataFrame(np.nan, index=mean.index, columns=mean.columns)
    if OUR_COLS:
        o = p.loc[OUR_COLS]
        mean.loc["ours"] = o.mean()
        std.loc["ours"] = o.std(ddof=0)
    mean = mean.reindex([r for r in ROW_IDS if r in mean.index])
    return mean, std.reindex(mean.index)


def table_png(df, title, path, bold=None):
    n_r, n_c = df.shape
    fig, ax = plt.subplots(figsize=(max(5.5, 1.45 * n_c + 2.6), 0.4 * n_r + 1.1))
    ax.axis("off")
    tb = ax.table(cellText=df.values, colLabels=list(df.columns), rowLabels=list(df.index), loc="center", cellLoc="center")
    tb.auto_set_font_size(False); tb.set_fontsize(9); tb.scale(1, 1.4)
    for (r, c), cell in tb.get_celld().items():
        cell.set_edgecolor("#d0d4da")
        if r == 0:
            cell.set_facecolor("#e8edf5"); cell.set_text_props(fontweight="bold")
        elif str(df.index[r - 1]).startswith("Ours"):
            cell.set_facecolor("#eef3fb")
        if bold is not None and r > 0 and c >= 0 and bold[r - 1][c]:
            cell.set_text_props(fontweight="bold")
    ax.set_title(title, fontweight="bold", fontsize=11, pad=8)
    fig.savefig(path, dpi=170, bbox_inches="tight"); plt.close(fig)


def md_table(df, bold=None):
    cols = ["Model"] + list(df.columns)
    lines = ["| " + " | ".join(cols) + " |", "|" + "|".join(["---"] * len(cols)) + "|"]
    for i, (idx, row) in enumerate(df.iterrows()):
        cells = [f"**{v}**" if (bold is not None and bold[i][j] and str(v) != "-") else str(v) for j, v in enumerate(row)]
        lines.append("| " + " | ".join([str(idx)] + cells) + " |")
    return "\n".join(lines)


def register(name, title, df, bold=None):
    TABLES[name] = df
    df.to_csv(OUT / f"slide_{name}.csv")
    table_png(df, title, FIG / f"table_{name}.png", bold)
    MD_PARTS.append(f"## {title}\n\n" + md_table(df, bold) + "\n")


def slide_table(name, title, mean, std, higher=True):
    cols = list(mean.columns)
    cells = pd.DataFrame({c: [fmt_cell(mean.loc[r, c], std.loc[r, c]) for r in mean.index] for c in cols}, index=mean.index)
    cells.insert(0, "Stage", [STAGE_OF.get(r, "-") for r in cells.index])
    cells["Rank"] = mean["Avg"].rank(ascending=not higher, method="min").astype(int).values
    best = mean.idxmax() if higher else mean.idxmin()
    bold = [[False] * (1 + len(cols) + 1) for _ in mean.index]
    for j, c in enumerate(cols):
        bold[list(mean.index).index(best[c])][1 + j] = True
    cells.index = [row_label(r) for r in cells.index]
    register(name, title, cells, bold)
    return cells


def wide_pooled(col):
    return PO.pivot(index="model_id", columns="benchmark", values=col)[BENCH]


def wide_macro(col):
    return MA.pivot(index="model_id", columns="benchmark", values=col)[BENCH]


def rename_cols(df, f=lambda b: BENCH_LABEL[b]):
    df = df.copy(); df.columns = [f(c) if c in BENCH_LABEL else c for c in df.columns]; return df


m_, s_ = with_ours(wide_pooled("auc")); m_, s_ = rename_cols(m_), rename_cols(s_)
T_MAIN = slide_table("auc_pooled", "AUC (pooled per benchmark) - Ours vs B-Free official vs baselines", m_, s_)
m_, s_ = with_ours(wide_macro("auc")); m_, s_ = rename_cols(m_), rename_cols(s_)
slide_table("auc_macro", "AUC (macro: mean over subsets/platforms)", m_, s_)
m_, s_ = with_ours(wide_pooled("bacc")); m_, s_ = rename_cols(m_), rename_cols(s_)
slide_table("bacc_thr0.5", "Balanced accuracy @ threshold 0.5 (pooled)", m_, s_)
m_, s_ = with_ours(wide_pooled("eer")); m_, s_ = rename_cols(m_), rename_cols(s_)
slide_table("eer_pooled", "EER (pooled, lower is better)", m_, s_, higher=False)
m_, s_ = with_ours(wide_pooled("ap")); m_, s_ = rename_cols(m_), rename_cols(s_)
slide_table("ap_pooled", "Average Precision (pooled)", m_, s_)

# TPR / TNR @ 0.5 (chuỗi 'TPR / TNR')
tp_m, _s = with_ours(wide_pooled("tpr_fake")); tn_m, _s = with_ours(wide_pooled("tnr_real"))
tt = pd.DataFrame({BENCH_LABEL[b]: [f"{tp_m.loc[r, b]:.2f} / {tn_m.loc[r, b]:.2f}" for r in tp_m.index] for b in BENCH}, index=tp_m.index)
tt.insert(0, "Stage", [STAGE_OF.get(r, "-") for r in tt.index])
tt.index = [row_label(r) for r in tt.index]
register("tpr_tnr_thr0.5", "TPR (fake) / TNR (real) @ threshold 0.5 (pooled)", tt)

# AUC từng subset (platform)
SUBSETS = [(b, g) for b in BENCH for g in sorted(PG[PG.benchmark == b].generator.unique())]
ps = PG.pivot_table(index="model_id", columns=["benchmark", "generator"], values="auc").reindex(columns=pd.MultiIndex.from_tuples(SUBSETS))
ps.columns = [f"{b}|{g}" for b, g in ps.columns]
m_, s_ = with_ours(ps)
SUB_LAB = {f"{b}|{g}": (BENCH_LABEL[b] if g == b else f"{BENCH_LABEL[b]} / {g}") for b, g in SUBSETS}
m_, s_ = m_.rename(columns=SUB_LAB), s_.rename(columns=SUB_LAB)
SUB_MEAN = m_.copy()
slide_table("auc_per_subset", "AUC per subset (platform)", m_.round(6), s_.round(6))

# Δ AUC (Ours - ref) với CI bootstrap cặp
SYM = {"ours better": "▲", "ours worse": "▼", "n.s.": "≈"}
if len(PB):
    for lvl, nm, ttl in [("pooled", "delta_auc_pooled", "ΔAUC pooled = Ours - ref, 95% paired-bootstrap CI  (▲ ours better, ▼ worse, ≈ n.s.)"),
                         ("macro", "delta_auc_macro", "ΔAUC macro = Ours - ref, 95% paired-bootstrap CI  (▲ ours better, ▼ worse, ≈ n.s.)")]:
        q = PB[PB.level == lvl].set_index(["ref", "benchmark"])
        d = pd.DataFrame({BENCH_LABEL[b]: [f"{q.loc[(r, b), 'd_auc']:+.{DEC}f} [{q.loc[(r, b), 'd_lo']:+.{DEC}f}, {q.loc[(r, b), 'd_hi']:+.{DEC}f}] {SYM[q.loc[(r, b), 'verdict']]}"
                                           for r in REF_COLS] for b in BENCH}, index=[label_of(r) for r in REF_COLS])
        d.insert(0, "Stage", [STAGE_OF.get(r, "-") for r in REF_COLS])
        register(nm, ttl, d)

# Provenance: model nào lấy từ stage/folder nào
prov = pd.DataFrame(PROV).groupby(["model_id", "stage", "folder"]).agg(files=("file", "nunique"), rows=("rows", "sum")).reset_index()
prov["in_main_table"] = prov.model_id.isin(MODELS)
prov["label"] = prov.model_id.map(label_of)
prov.to_csv(OUT / "provenance_models.csv", index=False)
print(prov.to_string(index=False))
print()
print(md_table(T_MAIN))


In [ ]:
# 6) Chi phí tính toán (giây GPU / ảnh, từ speed_report*.json nếu có) -> bảng efficiency
sp = {}
for root, _, files in os.walk(INPUT_ROOT):
    for f in files:
        if re.match(r"speed_report.*\.json$", f):
            try:
                data = json.loads((Path(root) / f).read_text())
            except Exception:
                continue
            for r in data if isinstance(data, list) else []:
                if r.get("benchmark") in BENCH and r.get("model_seconds"):
                    sp[(Path(root).name, f, r["benchmark"])] = r
COST = None
if sp:
    sec, imgs = {}, {}
    for (folder, f, b), r in sp.items():
        for m, s in r["model_seconds"].items():
            sec[(folder, m)] = sec.get((folder, m), 0.0) + float(s)
            imgs[(folder, m)] = imgs.get((folder, m), 0) + int(r["n_scored"])
    crow = {}
    for (folder, m), s in sec.items():
        crow.setdefault(m, []).append(s / imgs[(folder, m)])
    COST = pd.Series({m: float(np.mean(v)) for m, v in crow.items()})
    COST = COST[[m for m in COST.index if m in MODELS]]
    print("Giây GPU/ảnh (T4, cộng cả GPU, chỉ mang tính tương đối giữa các model):")
    print(COST.sort_values().round(4).to_string())
    ct = pd.DataFrame({"GPU-sec / image": [COST.get(r, np.nan) if r != "ours" else COST.reindex(OUR_COLS).mean() for r in ROW_IDS]}, index=ROW_IDS)
    ct["images / GPU-sec"] = 1 / ct["GPU-sec / image"]
    ptab = with_ours(wide_pooled("auc"))[0]
    ct["Avg pooled AUC"] = [ptab.loc[r, "Avg"] for r in ROW_IDS]
    cts = ct.copy()
    cts["GPU-sec / image"] = cts["GPU-sec / image"].map(lambda v: "-" if pd.isna(v) else f"{v:.3f}")
    cts["images / GPU-sec"] = cts["images / GPU-sec"].map(lambda v: "-" if pd.isna(v) else f"{v:.1f}")
    cts["Avg pooled AUC"] = cts["Avg pooled AUC"].map(lambda v: f"{v:.{DEC}f}")
    cts.index = [row_label(r) for r in cts.index]
    register("cost", "Inference cost (T4, GPU-seconds per image) vs average AUC", cts)
else:
    print("Không có speed_report*.json (hoặc không có model_seconds) -> bỏ qua bảng chi phí")


## 5. Biểu đồ

In [ ]:
# 7) Biểu đồ (A): AUC pooled theo benchmark, xếp hạng giữa các model
def save(fig, name):
    fig.savefig(FIG / f"{name}.png", dpi=170, bbox_inches="tight")
    plt.show()


ptab_m, ptab_s = with_ours(wide_pooled("auc"))
fig, axs = plt.subplots(1, len(BENCH), figsize=(5.4 * len(BENCH), 0.42 * len(ROW_IDS) + 1.7), squeeze=False)
lo_lim = max(0.3, float(np.nanmin(ptab_m[BENCH].values)) - 0.08)
for ax, b in zip(axs.ravel(), BENCH):
    v = ptab_m[b].sort_values()
    col = [C_OURS if r == "ours" else (C_OFF if r == OFFICIAL else C_BASE) for r in v.index]
    ax.barh(range(len(v)), v.values - lo_lim, left=lo_lim, color=col, height=0.66)
    if "ours" in v.index and pd.notna(ptab_s.loc["ours", b]):
        ax.errorbar(v["ours"], list(v.index).index("ours"), xerr=ptab_s.loc["ours", b], fmt="none", ecolor="white", capsize=3, lw=1.2)
    for i, (r, x) in enumerate(v.items()):
        ax.text(min(x + 0.004, 0.995), i, f"{x:.3f}", va="center", fontsize=8)
    ax.set_yticks(range(len(v))); ax.set_yticklabels([row_label(r).replace(" (mean ± std, ", " (mean, ") for r in v.index], fontsize=8.5)
    ax.axvline(0.5, color="gray", lw=0.8, ls=":")
    ax.set_xlim(lo_lim, 1.0); ax.set_title(BENCH_LABEL[b]); ax.set_xlabel("AUC (pooled)"); ax.grid(axis="y", alpha=0)
fig.suptitle("AUC per benchmark (blue = Ours, orange = B-Free official, grey = baselines)", fontweight="bold")
fig.tight_layout(rect=[0, 0, 1, 0.95])
save(fig, "auc_bars_by_benchmark")

# Heatmap AUC theo subset
hm = SUB_MEAN.copy()
hm = hm.drop(columns=["Avg"]) if "Avg" in hm.columns else hm
fig, ax = plt.subplots(figsize=(1.0 * hm.shape[1] + 3.2, 0.42 * hm.shape[0] + 1.8))
im = ax.imshow(hm.values.astype(float), cmap="RdYlGn", vmin=0.45, vmax=1.0, aspect="auto")
ax.set_xticks(range(hm.shape[1])); ax.set_xticklabels(hm.columns, rotation=35, ha="right", fontsize=8.5)
ax.set_yticks(range(hm.shape[0])); ax.set_yticklabels([row_label(r).replace(" (mean ± std, ", " (mean, ") for r in hm.index], fontsize=8.5)
for i in range(hm.shape[0]):
    for j in range(hm.shape[1]):
        ax.text(j, i, f"{hm.values[i, j]:.2f}", ha="center", va="center", fontsize=8)
ax.grid(False); ax.set_title("AUC per subset (platform)")
fig.colorbar(im, ax=ax, fraction=0.03, pad=0.02)
save(fig, "auc_heatmap_subsets")


In [ ]:
# 8) Biểu đồ (B): ΔAUC vs mỗi ref (forest), điểm vận hành @0.5, ROC, AUC vs chi phí
if len(PB):
    fig, axs = plt.subplots(1, len(BENCH), figsize=(5.2 * len(BENCH), 0.42 * len(REF_COLS) + 1.8), squeeze=False)
    for ax, b in zip(axs.ravel(), BENCH):
        q = PB[(PB.level == "pooled") & (PB.benchmark == b)].set_index("ref").reindex(REF_COLS).sort_values("d_auc")
        y = np.arange(len(q))
        cols = np.where(q.verdict == "ours better", "#2e7d4f", np.where(q.verdict == "ours worse", "#b3402f", "#8a8a8a"))
        ax.errorbar(q.d_auc, y, xerr=np.vstack([(q.d_auc - q.d_lo).clip(lower=0), (q.d_hi - q.d_auc).clip(lower=0)]), fmt="none", ecolor="black", lw=1, capsize=2)
        ax.scatter(q.d_auc, y, c=cols, s=40, zorder=3)
        ax.axvline(0, color="black", lw=0.8)
        ax.set_yticks(y); ax.set_yticklabels([label_of(r) for r in q.index], fontsize=8.5); ax.grid(axis="y", alpha=0)
        ax.set_title(BENCH_LABEL[b]); ax.set_xlabel("ΔAUC (Ours − ref), pooled")
    fig.suptitle("Gain of Ours over each reference (green/red: 95% paired CI excludes 0)", fontweight="bold")
    fig.tight_layout(rect=[0, 0, 1, 0.94])
    save(fig, "delta_auc_vs_refs")

# điểm vận hành @ ngưỡng 0.5: TNR(real) vs TPR(fake)
tp_m, _s = with_ours(wide_pooled("tpr_fake")); tn_m, _s = with_ours(wide_pooled("tnr_real"))
fig, axs = plt.subplots(1, len(BENCH), figsize=(5.2 * len(BENCH), 4.6), squeeze=False)
for ax, b in zip(axs.ravel(), BENCH):
    for r in tp_m.index:
        ax.scatter(tn_m.loc[r, b], tp_m.loc[r, b], s=70 if r in ("ours", OFFICIAL) else 45, color=MCOL.get(r, C_BASE),
                   marker="D" if r == OFFICIAL else "o", zorder=3, edgecolor="white")
        ax.annotate(row_label(r).split(" (")[0], (tn_m.loc[r, b], tp_m.loc[r, b]), xytext=(4, 3), textcoords="offset points", fontsize=7.5)
    ax.plot([0, 1], [1, 0], color="gray", lw=0.8, ls=":")
    ax.set_xlim(-0.03, 1.05); ax.set_ylim(-0.03, 1.05)
    ax.set_xlabel("TNR on real @0.5"); ax.set_ylabel("TPR on fake @0.5"); ax.set_title(BENCH_LABEL[b])
fig.suptitle("Operating point at the default threshold (top-right is best)", fontweight="bold")
fig.tight_layout(rect=[0, 0, 1, 0.94])
save(fig, "operating_point_thr0.5")

# ROC pooled: Ours (3 seed), official, top-3 baseline theo từng benchmark
fig, axs = plt.subplots(1, len(BENCH), figsize=(5.2 * len(BENCH), 4.4), squeeze=False)
for ax, b in zip(axs.ravel(), BENCH):
    d = S[S.benchmark == b]
    top = [m for m in sorted(BASE_IDS, key=lambda m: -POp.loc[b, m])[:3]] if b in POp.index else []
    for mid in OUR_COLS + ([OFFICIAL] if OFFICIAL in MODELS else []) + top:
        dm = d[d.model_id == mid]
        if dm.empty or dm.label.nunique() < 2:
            continue
        fpr, tpr, _ = roc_curve(dm.label.values, dm.logit.values)
        ax.plot(fpr, tpr, color=C_SEED[OUR_COLS.index(mid)] if mid in OUR_COLS else MCOL[mid], lw=1.1 if mid in OUR_COLS else 1.5,
                ls="--" if mid == OFFICIAL else "-", label=f"{label_of(mid)}  AUC={roc_auc_score(dm.label.values, dm.logit.values):.3f}")
    ax.set_xscale("log"); ax.set_xlim(1e-3, 1); ax.set_ylim(0, 1.01)
    ax.set_xlabel("FPR (log)"); ax.set_ylabel("TPR"); ax.set_title(f"{BENCH_LABEL[b]} (pooled)")
    ax.legend(frameon=False, fontsize=7.5, loc="lower right")
fig.tight_layout()
save(fig, "roc_pooled")

# AUC trung bình vs chi phí suy luận
if COST is not None and len(COST) >= 3:
    fig, ax = plt.subplots(figsize=(6.4, 4.4))
    for r in ROW_IDS:
        c = COST.reindex(OUR_COLS).mean() if r == "ours" else COST.get(r, np.nan)
        if pd.isna(c):
            continue
        ax.scatter(c, ptab_m.loc[r, "Avg"], s=80 if r in ("ours", OFFICIAL) else 50, color=MCOL.get(r, C_BASE),
                   marker="D" if r == OFFICIAL else "o", edgecolor="white", zorder=3)
        ax.annotate(row_label(r).split(" (")[0], (c, ptab_m.loc[r, "Avg"]), xytext=(5, 3), textcoords="offset points", fontsize=8)
    ax.set_xscale("log"); ax.set_xlabel("GPU-seconds per image (T4, log scale)"); ax.set_ylabel("Average pooled AUC")
    ax.set_title("Accuracy vs inference cost")
    save(fig, "auc_vs_cost")


## 6. Nhận xét tự động + xuất file

In [ ]:
# 9) slide_notes.md: các câu nhận xét sinh từ số liệu (kiểm tra lại trước khi đưa lên slide)
L = []
ptab_num = with_ours(wide_pooled("auc"))[0]
bacc_num = with_ours(wide_pooled("bacc"))[0]
n_models = len(ptab_num)
L.append(f"# Ghi chú cho slide\n\nNguồn: {', '.join(sorted(set(STAGE_OF[m] for m in MODELS)))} | {len(MODELS)} model đủ ảnh "
         f"(Ours = {len(OUR_COLS)} seed, B-Free official, {len(BASE_IDS)} baseline) | benchmark: {', '.join(BENCH_LABEL[b] for b in BENCH)}\n")
for b in BENCH:
    v = ptab_num[b].dropna()
    if "ours" not in v.index:
        continue
    rank = int(v.rank(ascending=False, method="min")["ours"])
    best_ref = v.drop("ours").idxmax()
    line = f"- **{BENCH_LABEL[b]}**: Ours AUC {v['ours']:.3f}, hạng {rank}/{len(v)}."
    line += f" Ref tốt nhất: {label_of(best_ref)} {v[best_ref]:.3f}"
    if len(PB):
        q = PB[(PB.level == "pooled") & (PB.benchmark == b) & (PB.ref == best_ref)]
        if len(q):
            q = q.iloc[0]
            line += f" (Δ = {q.d_auc:+.3f} [{q.d_lo:+.3f}, {q.d_hi:+.3f}] -> {q.verdict})"
    L.append(line + ".")
    if len(PB):
        qq = PB[(PB.level == "pooled") & (PB.benchmark == b)]
        L.append(f"  - So với {len(qq)} ref: ours better {int((qq.verdict == 'ours better').sum())}, "
                 f"n.s. {int((qq.verdict == 'n.s.').sum())}, ours worse {int((qq.verdict == 'ours worse').sum())}"
                 + (f"; Ours KHÔNG vượt có ý nghĩa: {', '.join(label_of(r) + ' (' + qq[qq.ref == r].verdict.iloc[0] + ')' for r in qq[qq.verdict != 'ours better'].ref)}" if (qq.verdict != 'ours better').any() else "") + ".")
avg = ptab_num["Avg"].sort_values(ascending=False)
L.append("\n**Xếp hạng AUC pooled trung bình**: " + " > ".join(f"{row_label(r).split(' (')[0]} {x:.3f}" for r, x in avg.items()))
miscal = [r for r in ptab_num.index if ptab_num.loc[r, "Avg"] >= 0.85 and (ptab_num.loc[r, "Avg"] - bacc_num.loc[r, "Avg"]) >= 0.10]
if miscal:
    L.append("\n**Lệch ngưỡng @0.5** (AUC cao nhưng bacc thấp hơn >= 0.10): " + ", ".join(
        f"{row_label(r).split(' (')[0]} (AUC {ptab_num.loc[r, 'Avg']:.3f} vs bacc {bacc_num.loc[r, 'Avg']:.3f})" for r in miscal)
        + " -> so sánh ở ngưỡng cố định không công bằng với các model này; dùng AUC/EER.")
chance = [r for r in ptab_num.index if r not in ("ours", OFFICIAL) and ptab_num.loc[r, "Avg"] < 0.65]
if chance:
    L.append("\n**Gần ngẫu nhiên (AUC trung bình < 0.65)**: " + ", ".join(
        f"{label_of(r)} {ptab_num.loc[r, 'Avg']:.3f}" for r in chance)
        + ". Trên ảnh mạng xã hội (nén JPEG, resize) đây có thể là mất khả năng tổng quát thật, nhưng cần loại trừ lỗi protocol (view/normalize/dtype) trước khi trình bày như kết luận.")
    for r in chance:
        if r in PROTOCOL_CAVEATS:
            L.append(f"  - {label_of(r)}: {PROTOCOL_CAVEATS[r]}")
if PARTIAL:
    L.append("\n**Chưa so sánh được** (chấm chưa đủ ảnh): " + ", ".join(label_of(m) for m in PARTIAL) + " - xem partial_models.csv.")
L.append("\n*CI của ΔAUC chỉ phản ánh biến động do chọn ảnh (3 seed lấy trung bình); độ lệch giữa các seed nằm ở ± std trong bảng.*")
(OUT / "slide_notes.md").write_text("\n".join(L) + "\n")
(OUT / "slide_tables.md").write_text("# Bảng cho slide\n\n" + "\n".join(MD_PARTS))
print("\n".join(L))

# 10) Excel gộp tất cả bảng
try:
    with pd.ExcelWriter(OUT / "slide_tables.xlsx", engine="openpyxl") as xw:
        for nm, df in TABLES.items():
            df.to_excel(xw, sheet_name=nm[:31])
            ws = xw.sheets[nm[:31]]
            ws.column_dimensions["A"].width = 34
            for col in ws.iter_cols(min_col=2, max_col=ws.max_column):
                ws.column_dimensions[col[0].column_letter].width = 26
        prov.to_excel(xw, sheet_name="provenance", index=False)
except Exception as e:
    print("Bỏ qua xuất Excel:", repr(e))

print("\nFile đã ghi:", sorted(p.name for p in list(OUT.glob("*.csv")) + list(OUT.glob("*.md")) + list(OUT.glob("*.xlsx")) + list(FIG.glob("*.png"))))